# Tenant Satisfaction Measures: Step 1, load and explore

I work in social housing, and every year the Regulator of Social Housing publishes the Tenant Satisfaction Measures (TSMs) for every large landlord in England. I wanted to use them to practise Python on data I actually understand, and to answer a question I'm genuinely curious about:

**What drives tenant satisfaction across England's large social landlords?**

**Data:** Regulator of Social Housing, *Tenant Satisfaction Measures 2025: full data set* (the year 1 April 2024 to 31 March 2025). It covers landlords with 1,000 or more homes.

In this first notebook I'm just getting to know the data: loading it, tidying the column names, seeing what kinds of landlords are in there, and checking for gaps.

## 1. Load the data

I wrote this to run in Google Colab, so the first cell pops up an upload button. If you run it on your own computer instead, it looks in the `data` folder.

In [1]:
import pandas as pd

# In Colab I upload the file with a button. On my own computer that import fails,
# so the "except" part kicks in and I point at the data folder instead.
try:
    from google.colab import files
    uploaded = files.upload()
    path = list(uploaded.keys())[0]
except ImportError:
    path = "../data/2025_TSM_Full_Data_v1.1_FINAL.xlsx"

In [2]:
# When I opened the file in Excel, the real column headers were on row 11, with notes above.
# pandas counts from 0, so row 11 in Excel is header=10.
# RSH also puts codes in some cells instead of leaving them blank ([X] left blank,
# [Z] not required, [E] excluded). na_values tells pandas to treat those as missing,
# otherwise the whole column would be read as text and I couldn't do maths on it.
lcra = pd.read_excel(path, sheet_name="TSM25_LCRA_Perception",
                     header=10, na_values=["[X]", "[Z]", "[E]"])

print(lcra.shape)  # (rows, columns), so I can see how big it is
lcra.iloc[:3, :6]

(353, 135)


,Landlord name,Landlord code,Landlord type,Landlord TSM Publication URL1,"Landlord group size (homes owned, LCRA and LCHO combined)",Landlord predominant region
0,A2Dominion Housing Group Limited,L4240,PRP,https://a2dominion.co.uk/about-us/tsm,"5,000-40,000",South
1,Abri Group Limited,L4172,PRP,www.abri.co.uk/customer-support/tenant-satisfaction-measures,">40,000",South
2,Acis Group Limited,L4229,PRP,https://www.acisgroup.co.uk/about-us/our-performance/tsms/,"5,000-40,000",Midlands


## 2. Shorten the column names

The first thing I noticed is that the column names are whole sentences, like *"...satisfied with the overall service provided by their landlord (TP01)"*. Some even have a footnote number stuck on the end, like *"(TP09)7"*. I definitely didn't want to type those out every time.

Every headline measure starts the same way and ends with its code in brackets, so I used a **regular expression** (a pattern for matching text) to find them and pull out the code. The pattern only matches names that *end* with the code, which conveniently skips the extra TP01 columns broken down by survey method.

In [3]:
# My pattern: starts with the standard wording, then anything (.*), then a code like (TP01).
# (TP\d{2}) "captures" the code so I can use it as the new name. \d*$ allows a footnote number at the end.
tp_pattern = r"^Proportion of respondents who report that they are satisfied.*\((TP\d{2})\)\d*$"

# Go through every column and, if it matches, remember {long name: short code}.
rename_map = {}
for col in lcra.columns:
    match = pd.Series([col]).str.extract(tp_pattern).iloc[0, 0]
    if pd.notna(match):
        rename_map[col] = match

# I also gave short names to the other columns I'll use a lot.
# (The tenants column has a sneaky non-breaking space in its name, hence the \xa0.)
rename_map.update({
    "Landlord name": "landlord",
    "Landlord code": "code",
    "Landlord type": "type",
    "Landlord group size (homes owned, LCRA and LCHO combined)": "size_band",
    "Landlord predominant region": "region",
    "LCRA above 30% supported housing": "supported_30pc",
    "Relevant\xa0tenant population size (LCRA)": "tenants",
    "Total sample size achieved": "sample_size",
    "Predominant collection method": "method",
})

# Check I found all 12 measures.
tp_cols = sorted(c for c in rename_map.values() if c.startswith("TP"))
print(len(tp_cols), "perception measures found:", tp_cols)

# Keep just the columns I want, with their new names, in a new table called df.
keep = ["landlord", "code", "type", "size_band", "region", "supported_30pc",
        "tenants", "sample_size", "method"] + tp_cols
df = lcra.rename(columns=rename_map)[keep].copy()
df.head()

12 perception measures found: ['TP01', 'TP02', 'TP03', 'TP04', 'TP05', 'TP06', 'TP07', 'TP08', 'TP09', 'TP10', 'TP11', 'TP12']


,landlord,code,type,size_band,region,supported_30pc,tenants,sample_size,method,TP01,TP02,TP03,TP04,TP05,TP06,TP07,TP08,TP09,TP10,TP11,TP12
0,A2Dominion Housing Group Limited,L4240,PRP,"5,000-40,000",South,No,20780,1317,Telephone,60.7,63.7,61.1,63.8,71.8,54.6,66.9,69.3,30.9,61.5,56.8,55.6
1,Abri Group Limited,L4172,PRP,">40,000",South,No,34460,2433,Telephone,74.5,76.2,69.9,74.6,81.6,65.2,75.0,79.6,41.3,66.6,65.3,58.8
2,Acis Group Limited,L4229,PRP,"5,000-40,000",Midlands,No,6085,982,Telephone,80.4,84.7,84.1,80.5,86.6,70.9,83.3,86.1,53.0,71.2,73.6,63.6
3,Anchor Hanover Group,LH4095,PRP,"5,000-40,000",Mixed,Yes,35759,2851,Telephone,86.1,86.2,81.1,87.2,91.2,74.9,88.8,89.0,41.6,90.7,82.5,76.4
4,Arawak Walton Housing Association Limited,L3713,PRP,"<5,000",North,No,1080,301,Telephone,84.3,85.0,83.7,84.0,86.4,83.5,94.8,92.4,46.3,83.2,84.7,80.1


## 3. What each measure means

A note for myself, since I'll be using the codes from now on:

| Code | Satisfaction with... |
| --- | --- |
| TP01 | Overall service |
| TP02 | Overall repairs service |
| TP03 | Time taken to complete most recent repair |
| TP04 | Home is well maintained |
| TP05 | Home is safe |
| TP06 | Landlord listens to views and acts on them |
| TP07 | Landlord keeps tenants informed |
| TP08 | Landlord treats tenants fairly and with respect |
| TP09 | Landlord's approach to complaints handling |
| TP10 | Communal areas kept clean and well maintained |
| TP11 | Landlord makes a positive contribution to the neighbourhood |
| TP12 | Landlord's approach to anti-social behaviour |

## 4. Profile the landlords

Before doing anything clever, I wanted to know what kinds of landlords are in the data: housing associations or councils, how big they are, how they ran their surveys, and where they are.

In [4]:
# value_counts() counts how many landlords fall in each group.
# PRP = private registered provider (a housing association), LA = local authority (a council).
print(df["type"].value_counts(), "\n")
print(df["size_band"].value_counts(), "\n")
print(df["method"].value_counts(), "\n")
print(df["region"].value_counts())

type
PRP    195
LA     158
Name: count, dtype: int64 

size_band
5,000-40,000    212
<5,000          122
>40,000          19
Name: count, dtype: int64 

method
Telephone                242
Internet                  44
Postal                    29
No predominant method     18
Face to Face              15
SMS                        5
Name: count, dtype: int64 

region
South       132
North        91
Midlands     64
London       52
Mixed        14
Name: count, dtype: int64


In [5]:
# describe() gives the count, average, spread and range for each measure.
# .T flips the table on its side so each measure is a row, which is easier to read.
df[tp_cols].describe().T.round(1)

,count,mean,std,min,25%,50%,75%,max
TP01,353.0,71.3,10.0,43.8,64.7,71.8,78.9,97.5
TP02,353.0,73.1,8.9,44.9,66.8,73.6,79.6,96.9
TP03,353.0,69.4,9.7,43.1,63.0,69.5,76.5,96.1
TP04,353.0,71.2,9.6,41.4,65.5,71.9,77.8,96.6
TP05,353.0,77.1,8.3,46.7,71.8,77.6,82.9,98.1
TP06,353.0,61.5,10.9,32.3,54.1,61.6,69.3,95.3
TP07,353.0,71.4,9.5,40.4,65.9,72.0,77.3,94.8
TP08,353.0,77.3,8.7,45.5,72.6,77.9,83.9,97.2
TP09,352.0,36.4,10.3,13.8,29.6,35.5,42.1,94.3
TP10,352.0,66.4,9.2,41.2,60.7,66.6,72.8,97.4


In [6]:
# I wondered whether the way a survey is done affects the scores, so I grouped landlords
# by their main survey method and looked at the median overall satisfaction for each.
(df.groupby("method")["TP01"]
   .agg(landlords="count", median_TP01="median")
   .sort_values("median_TP01", ascending=False)
   .round(1))

,landlords,median_TP01
method,,
Postal,29,77.6
Telephone,242,71.6
Face to Face,15,71.0
No predominant method,18,70.8
SMS,5,70.6
Internet,44,67.9


## 5. Check for missing values

Gaps can quietly mess up averages and models later, so I checked for them now.

In [7]:
# isna() marks each missing cell as True, and sum() counts the Trues in each column.
missing = df[tp_cols].isna().sum()
print(missing[missing > 0])
print("Total missing cells:", missing.sum(), "out of", df[tp_cols].size)

TP09    1
TP10    1
dtype: int64
Total missing cells: 2 out of 4236


In [8]:
# Which landlords have the gaps? any(axis=1) picks rows with at least one missing measure.
gaps = df[df[tp_cols].isna().any(axis=1)]
gaps[["landlord", "type", "size_band"] + [c for c in tp_cols if df[c].isna().any()]]

,landlord,type,size_band,TP09,TP10
241,Your Housing Group Limited,PRP,"5,000-40,000",45.3,NaN
351,St Mungo Community Housing Association,PRP,"<5,000",NaN,88.2


Hardly anything is missing: just 2 cells out of 4,236, because landlords have to report all 12 measures. Both gaps are in TP09 (complaints) and TP10 (communal areas). Those are the two questions only some tenants are asked (people who have complained, or who live somewhere with communal areas), so my guess is that's why, although the data doesn't say. I've left them as missing rather than making up a value.

## 6. Which measures vary most between landlords?

If every landlord scores about the same on something, it can't tell me much about why some landlords do better than others. So I looked at how spread out each measure is, using the gap between a landlord at the 25th percentile and one at the 75th (the **interquartile range**, or IQR).

In [9]:
# The middle 50% of landlords on each measure: 25th percentile, median and 75th percentile.
spread = pd.DataFrame({
    "25th percentile": df[tp_cols].quantile(0.25),
    "median": df[tp_cols].median(),
    "75th percentile": df[tp_cols].quantile(0.75),
})
# The IQR: how far apart a typical lower-scoring and a typical higher-scoring landlord are.
spread["IQR (points)"] = spread["75th percentile"] - spread["25th percentile"]
spread.sort_values("IQR (points)", ascending=False).round(1)

,25th percentile,median,75th percentile,IQR (points)
TP06,54.1,61.6,69.3,15.2
TP01,64.7,71.8,78.9,14.2
TP11,57.7,64.6,71.7,14.0
TP03,63.0,69.5,76.5,13.5
TP02,66.8,73.6,79.6,12.8
TP09,29.6,35.5,42.1,12.5
TP04,65.5,71.9,77.8,12.3
TP10,60.7,66.6,72.8,12.1
TP12,54.7,59.5,66.4,11.7
TP07,65.9,72.0,77.3,11.4


**Listens and acts (TP06) varies most between landlords** (15.2 points between the 25th and 75th percentile), followed by overall satisfaction (14.2) and positive contribution to the neighbourhood (14.0). Feeling safe (TP05) and being treated fairly (TP08) vary least, at about 11 points.

**Complaints handling (TP09) stands out for a different reason:** its median is just 35.5%, far below every other measure. Even at a typical landlord, most tenants who have complained aren't satisfied with how it was handled.

## What I found in step 1

- There are **353 landlords**: 195 housing associations and 158 councils. Most (212) have between 5,000 and 40,000 homes.
- **Telephone surveys are by far the most common** (242 landlords). The survey method seems to make a real difference: landlords that mainly surveyed by post have a median overall satisfaction of 77.6%, against 67.9% for online. That's nearly 10 points, so I'll need to compare like with like.
- The data is **almost complete**: only 2 missing cells, both in questions that only some tenants are asked.
- **Listens and acts varies most between landlords**, and **complaints handling has by far the lowest scores** across the sector.

**Next:** in step 2 I'll add the management information (repairs, complaints, anti-social behaviour) and see which of those measures move with satisfaction.